# BiGRU Feature Importance — Shapley-style Analysis (56 Features)

**Goal:** Find which features actually matter and who helps who.

**Method:** Train on 400 random feature subsets, measure IoU for each.
Then compute per-feature marginal contribution and pairwise interactions.

- 400 random subsets, sizes 15–45
- 1 seed, 15 epochs per eval (fast — ranking, not absolute scores)
- Per feature: mean IoU(present) - mean IoU(absent) = marginal value
- Pairwise: synergy matrix for top features

**Output:** Ranked features + interaction matrix → smart starting set for targeted search.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json, time
from datetime import datetime

DATA_DIR = Path('../data')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
all_df = pd.read_csv(DATA_DIR / 'features.csv')
print(f'Loaded {len(all_df)} rows, {all_df["page_id"].nunique()} pages')

TAG_MAP = {'body': 0, 'header': 1, 'nav': 2, 'main': 3, 'article': 4,
           'section': 5, 'footer': 6, 'aside': 7, 'form': 8}
all_df['parent_tag_idx'] = all_df['parent_tag'].map(TAG_MAP).fillna(0).astype(int)

tag_trans, dist_head = [], []
for _, group in all_df.groupby('page_id', sort=False):
    transitions = (group['parent_tag'] != group['parent_tag'].shift(1)).astype(int)
    transitions.iloc[0] = 0
    tag_trans.append(transitions)
    last_h = -1
    dists = []
    for i, (_, row) in enumerate(group.iterrows()):
        if row['is_heading'] == 1:
            last_h = i
        dists.append(i - last_h if last_h >= 0 else i)
    dist_head.append(pd.Series(dists, index=group.index))

all_df['tag_transition'] = pd.concat(tag_trans)
all_df['dist_to_heading'] = pd.concat(dist_head)

all_pages = all_df['page_id'].unique()
rng = np.random.RandomState(42)
rng.shuffle(all_pages)
split_idx = int(len(all_pages) * 0.8)
train_ids = set(all_pages[:split_idx])
test_ids = set(all_pages[split_idx:])

train_df = all_df[all_df['page_id'].isin(train_ids)].reset_index(drop=True)
test_df = all_df[all_df['page_id'].isin(test_ids)].reset_index(drop=True)
print(f'Train: {len(train_df)} rows, {len(train_ids)} pages')
print(f'Test:  {len(test_df)} rows, {len(test_ids)} pages')

In [ ]:
ALL_FEATURES = [
    'position_pct', 'total_lines', 'depth',
    'ancestor_depth_ratio', 'parent_tag_idx',
    'in_header', 'in_nav', 'in_main', 'in_article',
    'in_footer', 'in_aside', 'in_form',
    'text_length', 'word_count', 'link_ratio',
    'is_link_only', 'is_heading', 'heading_level',
    'has_bold', 'is_list_item',
    'is_link_summary', 'is_cookie_summary',
    'punctuation_ratio', 'sentence_count', 'avg_word_length',
    'uppercase_ratio', 'number_ratio', 'type_token_ratio',
    'comma_count',
    'has_positive_class', 'has_negative_class', 'has_unlikely_class',
    'has_image', 'has_input',
    'is_button', 'has_aria_hidden', 'in_table', 'in_details',
    'mean_idf', 'max_idf', 'line_frequency',
    'word_novelty', 'word_novelty_sum',
    'cumulative_text_pct',
    'mean_text_length_w5', 'mean_link_ratio_w5',
    'block_size', 'block_text_density', 'block_link_density',
    'relative_position_in_block', 'sibling_text_variance',
    'style_group_size', 'style_group_link_density', 'style_group_mean_words',
    'tag_transition', 'dist_to_heading',
]

N_ALL = len(ALL_FEATURES)
missing = [f for f in ALL_FEATURES if f not in all_df.columns]
assert not missing, f'Missing: {missing}'
assert N_ALL == 56
print(f'Feature pool: {N_ALL}')

## Model infrastructure

In [ ]:
HIDDEN = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 5e-4
BATCH_SIZE = 16
EVAL_EPOCHS = 15
EVAL_SEED = 42

N_SAMPLES = 400
MIN_SUBSET = 15
MAX_SUBSET = 45

n_content = train_df['label'].sum()
POS_WEIGHT = (len(train_df) - n_content) / n_content
print(f'Pos weight: {POS_WEIGHT:.3f}')


class PageDataset(Dataset):
    def __init__(self, pages):
        self.pages = pages
    def __len__(self):
        return len(self.pages)
    def __getitem__(self, idx):
        feats, labels, _, _, _ = self.pages[idx]
        return feats, labels


def collate_fn(batch):
    feats_list, labels_list = zip(*batch)
    max_len = max(f.shape[0] for f in feats_list)
    n_feats = feats_list[0].shape[1]
    bs = len(batch)
    feats_padded = torch.zeros(bs, max_len, n_feats)
    labels_padded = torch.zeros(bs, max_len)
    mask = torch.zeros(bs, max_len, dtype=torch.bool)
    for i, (f, l) in enumerate(zip(feats_list, labels_list)):
        seq_len = f.shape[0]
        feats_padded[i, :seq_len] = f
        labels_padded[i, :seq_len] = l
        mask[i, :seq_len] = True
    return feats_padded, labels_padded, mask


class BiGRUModel(nn.Module):
    def __init__(self, n_features, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(n_features, hidden)
        self.gru = nn.GRU(
            hidden, hidden // 2, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = F.relu(self.proj(x))
        x, _ = self.gru(x)
        x = self.drop(x)
        return self.head(x).squeeze(2)


def compute_loss(logits, labels, mask):
    bce = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
    weights = torch.where(labels == 1, POS_WEIGHT, 1.0)
    bce_loss = (bce * weights * mask.float()).sum() / mask.sum()
    probs = torch.sigmoid(logits)
    probs_m = probs * mask.float()
    labels_m = labels * mask.float()
    inter = (probs_m * labels_m).sum(dim=1)
    union = (probs_m + labels_m - probs_m * labels_m).sum(dim=1)
    iou = (inter + 1) / (union + 1)
    return bce_loss + (1 - iou.mean())


def df_to_pages(df, feature_cols, feat_mean, feat_std):
    pages = []
    for pid, group in df.groupby('page_id', sort=False):
        group = group.sort_values('line_num')
        feats = (group[feature_cols].values.astype(np.float32) - feat_mean) / feat_std
        labels = group['label'].values.astype(np.float32)
        pages.append((torch.from_numpy(feats), torch.from_numpy(labels),
                       group['line_num'].values, group['span_lines'].values, pid))
    return pages


def eval_iou(model, test_pages):
    model.eval()
    ious = []
    with torch.no_grad():
        for feats, labels, line_nums, span_lines, _ in test_pages:
            logits = model(feats.unsqueeze(0).to(DEVICE))
            probs = torch.sigmoid(logits[0]).cpu().numpy()
            pred_bin = (probs >= 0.5).astype(int)
            true_bin = labels.numpy().astype(int)
            pred_set, truth_set = set(), set()
            for j in range(len(pred_bin)):
                ln, span = int(line_nums[j]), int(span_lines[j])
                lines = set(range(ln, ln + span))
                if pred_bin[j]: pred_set |= lines
                if true_bin[j]: truth_set |= lines
            if not pred_set and not truth_set:
                ious.append(1.0)
            else:
                inter = len(pred_set & truth_set)
                union = len(pred_set | truth_set)
                ious.append(inter / union if union else 0.0)
    return np.mean(ious)


def run_fast(feature_cols):
    """Single-seed, EVAL_EPOCHS training. Returns best IoU."""
    torch.manual_seed(EVAL_SEED)
    np.random.seed(EVAL_SEED)

    train_feats = train_df[feature_cols].values.astype(np.float32)
    feat_mean = train_feats.mean(axis=0)
    feat_std = train_feats.std(axis=0)
    feat_std[feat_std == 0] = 1.0

    train_pages = df_to_pages(train_df, feature_cols, feat_mean, feat_std)
    test_pages = df_to_pages(test_df, feature_cols, feat_mean, feat_std)

    train_loader = DataLoader(
        PageDataset(train_pages), batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collate_fn,
        generator=torch.Generator().manual_seed(EVAL_SEED),
    )

    model = BiGRUModel(len(feature_cols), hidden=HIDDEN,
                       num_layers=NUM_LAYERS, dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EVAL_EPOCHS)

    best_iou = 0.0
    for epoch in range(1, EVAL_EPOCHS + 1):
        model.train()
        for feats, labels, mask in train_loader:
            feats, labels, mask = feats.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
            logits = model(feats)
            loss = compute_loss(logits, labels, mask)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        scheduler.step()
        if epoch % 5 == 0 or epoch == EVAL_EPOCHS:
            iou = eval_iou(model, test_pages)
            if iou > best_iou:
                best_iou = iou
    return best_iou


print('run_fast() ready (1 seed, 15 epochs).')

## Run 400 random subsets

In [ ]:
# Generate random subsets
subset_rng = np.random.RandomState(2026)
subsets = []
for i in range(N_SAMPLES):
    n = subset_rng.randint(MIN_SUBSET, MAX_SUBSET + 1)
    idx = subset_rng.choice(N_ALL, size=n, replace=False)
    feats = [ALL_FEATURES[j] for j in sorted(idx)]
    subsets.append(feats)

sizes = [len(s) for s in subsets]
print(f'Generated {len(subsets)} subsets')
print(f'  Size range: {min(sizes)}-{max(sizes)}')
print(f'  Mean size: {np.mean(sizes):.1f}')

# Check feature coverage
from collections import Counter
feat_counts = Counter()
for s in subsets:
    feat_counts.update(s)
min_count = min(feat_counts.values())
max_count = max(feat_counts.values())
print(f'  Feature appearances: {min_count}-{max_count} (mean {np.mean(list(feat_counts.values())):.0f})')

In [ ]:
# Resume support: load previous results if they exist
RESULTS_PATH = DATA_DIR / 'feature_importance_56_results.json'

results = []
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        saved = json.load(f)
    results = saved.get('results', [])
    print(f'Resumed: {len(results)}/{N_SAMPLES} already done')
else:
    print(f'Starting fresh: {N_SAMPLES} evaluations')

t_total = time.time()

for i in range(len(results), N_SAMPLES):
    feats = subsets[i]
    t0 = time.time()
    iou = run_fast(feats)
    elapsed = time.time() - t0

    # Store as bitmask for compact storage
    mask = [1 if f in feats else 0 for f in ALL_FEATURES]
    results.append({'idx': i, 'iou': round(iou, 4), 'n_features': len(feats),
                    'mask': mask, 'time_s': round(elapsed, 1)})

    if (i + 1) % 10 == 0 or i == N_SAMPLES - 1:
        # Save progress
        with open(RESULTS_PATH, 'w') as f:
            json.dump({'timestamp': datetime.now().isoformat(),
                       'features': ALL_FEATURES, 'results': results}, f)
        elapsed_total = time.time() - t_total
        eta = elapsed_total / (i + 1 - (len(results) - len(results))) * (N_SAMPLES - i - 1) if i > 0 else 0
        mean_iou = np.mean([r['iou'] for r in results])
        print(f'  [{i+1:3d}/{N_SAMPLES}] IoU={iou:.4f}  n={len(feats):2d}  '
              f'mean={mean_iou:.4f}  {elapsed:.0f}s  '
              f'elapsed={elapsed_total/60:.0f}m')

total_time = time.time() - t_total
print(f'\nDone: {len(results)} evaluations in {total_time/60:.1f} min')

## Analysis: Marginal contribution per feature

In [ ]:
# For each feature: mean IoU when present vs absent
ious = np.array([r['iou'] for r in results])
masks = np.array([r['mask'] for r in results])  # (N_SAMPLES, 56)

print(f'Overall: mean IoU={ious.mean():.4f}, std={ious.std():.4f}')
print(f'  Best subset:  IoU={ious.max():.4f} ({results[ious.argmax()]["n_features"]} features)')
print(f'  Worst subset: IoU={ious.min():.4f} ({results[ious.argmin()]["n_features"]} features)')
print()

marginal = {}
for fi, feat in enumerate(ALL_FEATURES):
    present = masks[:, fi] == 1
    absent = masks[:, fi] == 0
    iou_present = ious[present].mean()
    iou_absent = ious[absent].mean()
    n_present = present.sum()
    n_absent = absent.sum()
    delta = iou_present - iou_absent
    marginal[feat] = {'delta': delta, 'iou_present': iou_present,
                      'iou_absent': iou_absent, 'n_present': int(n_present),
                      'n_absent': int(n_absent)}

ranked = sorted(marginal.items(), key=lambda x: x[1]['delta'], reverse=True)

print(f'{"Feature":30s} {"Delta":>8s} {"Present":>8s} {"Absent":>8s} {"N_pres":>6s}')
print('-' * 68)
for feat, v in ranked:
    print(f'{feat:30s} {v["delta"]:+8.4f} {v["iou_present"]:8.4f} {v["iou_absent"]:8.4f} {v["n_present"]:6d}')

In [ ]:
# Visualize marginal contributions
import matplotlib.pyplot as plt

feats_sorted = [f for f, _ in ranked]
deltas = [v['delta'] for _, v in ranked]
colors = ['green' if d > 0.002 else 'red' if d < -0.002 else 'gray' for d in deltas]

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(range(len(feats_sorted)), deltas, color=colors)
ax.set_yticks(range(len(feats_sorted)))
ax.set_yticklabels(feats_sorted, fontsize=8)
ax.set_xlabel('Marginal IoU contribution')
ax.set_title('Feature Marginal Value (Shapley-style, 400 random subsets)')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_DIR / 'feature_marginal_56.png', dpi=150)
plt.show()
print('Saved feature_marginal_56.png')

## Analysis: Pairwise interactions (top 30 features)

In [ ]:
# For top 30 features, compute pairwise synergy:
# synergy(A,B) = IoU(both) - IoU(A only) - IoU(B only) + IoU(neither)
# Positive = synergistic (better together), Negative = redundant

top_n = 30
top_feats = [f for f, _ in ranked[:top_n]]
top_idx = [ALL_FEATURES.index(f) for f in top_feats]

synergy = np.zeros((top_n, top_n))

for ai, fi_a in enumerate(top_idx):
    for bi, fi_b in enumerate(top_idx):
        if ai >= bi:
            continue
        a_on = masks[:, fi_a] == 1
        b_on = masks[:, fi_b] == 1

        both = a_on & b_on
        a_only = a_on & ~b_on
        b_only = ~a_on & b_on
        neither = ~a_on & ~b_on

        # Need enough samples in each quadrant
        if both.sum() < 10 or a_only.sum() < 10 or b_only.sum() < 10 or neither.sum() < 10:
            continue

        s = ious[both].mean() - ious[a_only].mean() - ious[b_only].mean() + ious[neither].mean()
        synergy[ai, bi] = s
        synergy[bi, ai] = s

# Show strongest synergies and redundancies
pairs = []
for ai in range(top_n):
    for bi in range(ai + 1, top_n):
        if synergy[ai, bi] != 0:
            pairs.append((top_feats[ai], top_feats[bi], synergy[ai, bi]))

pairs.sort(key=lambda x: x[2], reverse=True)

print('Top 15 SYNERGISTIC pairs (better together):')
print(f'{"Feature A":25s} {"Feature B":25s} {"Synergy":>8s}')
print('-' * 62)
for a, b, s in pairs[:15]:
    print(f'{a:25s} {b:25s} {s:+8.4f}')

print(f'\nTop 15 REDUNDANT pairs (worse together):')
print(f'{"Feature A":25s} {"Feature B":25s} {"Synergy":>8s}')
print('-' * 62)
for a, b, s in pairs[-15:]:
    print(f'{a:25s} {b:25s} {s:+8.4f}')

In [ ]:
# Heatmap of pairwise interactions
fig, ax = plt.subplots(figsize=(14, 12))
vmax = max(abs(synergy.min()), abs(synergy.max()))
im = ax.imshow(synergy, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(top_n))
ax.set_yticks(range(top_n))
ax.set_xticklabels(top_feats, rotation=90, fontsize=7)
ax.set_yticklabels(top_feats, fontsize=7)
ax.set_title('Pairwise Feature Synergy (blue=synergistic, red=redundant)')
plt.colorbar(im, ax=ax, label='Synergy score')
plt.tight_layout()
plt.savefig(DATA_DIR / 'feature_synergy_56.png', dpi=150)
plt.show()
print('Saved feature_synergy_56.png')

## Build optimal starting set

In [ ]:
# Strategy: take all features with positive marginal contribution,
# then check if any negative-marginal features have strong synergies
# with included features (rescue them if so).

positive_feats = [f for f, v in ranked if v['delta'] > 0]
negative_feats = [f for f, v in ranked if v['delta'] <= 0]

print(f'Positive marginal ({len(positive_feats)}): {positive_feats}')
print(f'Negative marginal ({len(negative_feats)}): {negative_feats}')

# Check if any negative features have strong synergy with positive ones
rescued = []
for feat in negative_feats:
    if feat not in top_feats:
        continue
    fi = top_feats.index(feat)
    best_synergy = max(synergy[fi, :]) if synergy[fi, :].any() else 0
    if best_synergy > abs(marginal[feat]['delta']):
        partner_idx = np.argmax(synergy[fi, :])
        partner = top_feats[partner_idx]
        rescued.append((feat, partner, best_synergy, marginal[feat]['delta']))
        print(f'  Rescue {feat}: synergy with {partner} ({best_synergy:+.4f}) > marginal harm ({marginal[feat]["delta"]:+.4f})')

smart_start = positive_feats + [r[0] for r in rescued]
print(f'\nSmart starting set: {len(smart_start)} features')
print(f'  Included: {sorted(smart_start)}')
print(f'  Excluded: {sorted(set(ALL_FEATURES) - set(smart_start))}')

In [ ]:
# Quick validation: train the smart start set with 3 seeds
from functools import partial

def run_3seed(feature_cols, num_epochs=20):
    ious = []
    for seed in [42, 123, 777]:
        torch.manual_seed(seed)
        np.random.seed(seed)
        train_feats = train_df[feature_cols].values.astype(np.float32)
        feat_mean = train_feats.mean(axis=0)
        feat_std = train_feats.std(axis=0)
        feat_std[feat_std == 0] = 1.0
        train_pages = df_to_pages(train_df, feature_cols, feat_mean, feat_std)
        test_pages = df_to_pages(test_df, feature_cols, feat_mean, feat_std)
        train_loader = DataLoader(
            PageDataset(train_pages), batch_size=BATCH_SIZE,
            shuffle=True, collate_fn=collate_fn,
            generator=torch.Generator().manual_seed(seed),
        )
        model = BiGRUModel(len(feature_cols), hidden=HIDDEN,
                           num_layers=NUM_LAYERS, dropout=DROPOUT).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
        best = 0.0
        for epoch in range(1, num_epochs + 1):
            model.train()
            for feats, labels, mask in train_loader:
                feats, labels, mask = feats.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
                logits = model(feats)
                loss = compute_loss(logits, labels, mask)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            scheduler.step()
            if epoch % 5 == 0:
                iou = eval_iou(model, test_pages)
                if iou > best:
                    best = iou
        ious.append(best)
    return np.mean(ious), np.std(ious), ious

print(f'Validating smart start ({len(smart_start)} features) with 3 seeds, 20 epochs...')
t0 = time.time()
mean_iou, std_iou, seed_ious = run_3seed(smart_start)
print(f'  IoU={mean_iou:.4f} +/- {std_iou:.4f}  seeds={[round(v,4) for v in seed_ious]}  ({time.time()-t0:.0f}s)')

print(f'\nValidating ALL 56 features for comparison...')
t0 = time.time()
mean_all, std_all, seed_all = run_3seed(ALL_FEATURES)
print(f'  IoU={mean_all:.4f} +/- {std_all:.4f}  seeds={[round(v,4) for v in seed_all]}  ({time.time()-t0:.0f}s)')

print(f'\nDelta: {mean_iou - mean_all:+.4f} ({len(smart_start)} vs 56 features)')

## Save results

In [ ]:
final_path = DATA_DIR / 'feature_importance_56_final.json'
with open(final_path, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'n_samples': N_SAMPLES,
        'eval_epochs': EVAL_EPOCHS,
        'marginal_ranking': [(f, round(v['delta'], 4)) for f, v in ranked],
        'positive_features': positive_feats,
        'negative_features': negative_feats,
        'rescued_features': [(r[0], r[1], round(r[2], 4)) for r in rescued],
        'smart_start': sorted(smart_start),
        'smart_start_iou': round(mean_iou, 4),
        'all_56_iou': round(mean_all, 4),
        'top_synergies': [(a, b, round(s, 4)) for a, b, s in pairs[:20]],
        'top_redundancies': [(a, b, round(s, 4)) for a, b, s in pairs[-20:]],
    }, f, indent=2)
print(f'Saved to {final_path}')

print(f'\n{"=" * 70}')
print(f'SUMMARY')
print(f'{"=" * 70}')
print(f'  All 56:     IoU={mean_all:.4f}')
print(f'  Smart {len(smart_start):2d}:   IoU={mean_iou:.4f} ({mean_iou - mean_all:+.4f})')
print(f'  Dropped: {sorted(set(ALL_FEATURES) - set(smart_start))}')
print(f'\nUse smart_start as input to backward/forward search for final optimization.')